# 01. Computer Use 与 GUI Agent

Computer Use Agent 是 Agent 技术的新前沿：Agent 不仅调用 API，而是像人一样「看屏幕、点按钮、敲键盘」。

核心 pipeline：
```
Screenshot → Vision Model → Element Detection → Action (click/type/scroll) → New Screenshot
```

> **检查点**：能说出 Computer Use Agent 比 API-based Agent 多出的安全风险。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# Toy GUI Agent: 基于文本 UI 表示的 GUI 操作
from dataclasses import dataclass

@dataclass
class UIElement:
    id: str; type: str; label: str; row: int; col: int; width: int = 10

class ToyScreen:
    """用文本表示一个简单的 UI 界面"""
    def __init__(self, app_name: str = "File Manager"):
        self.app = app_name
        self.elements = [
            UIElement("btn_new", "button", "New File", 0, 1),
            UIElement("btn_open", "button", "Open", 0, 12),
            UIElement("btn_delete", "button", "Delete", 0, 22),
            UIElement("input_search", "input", "Search...", 2, 1, 30),
            UIElement("list_files", "list", "file1.txt\nfile2.pdf", 4, 1, 40),
        ]
    
    def find_element(self, label: str) -> UIElement | None:
        for e in self.elements:
            if label.lower() in e.label.lower():
                return e
        return None
    
    def render(self) -> str:
        lines = [f"[ {self.app} ]"]
        for e in self.elements:
            lines.append(f"  [{e.id}] {e.label} @ ({e.row},{e.col})")
        return "\n".join(lines)

class ToyGUIAgent:
    def __init__(self, screen: ToyScreen):
        self.screen = screen; self.log = []
    
    def decide_action(self, goal: str) -> str:
        """根据 goal 和当前屏幕状态决定操作"""
        if "new" in goal.lower() or "create" in goal.lower():
            return "click btn_new"
        if "delete" in goal.lower() or "remove" in goal.lower():
            return "click btn_delete"
        if "search" in goal.lower() or "find" in goal.lower():
            return "type input_search 'target_file'"
        return "done"
    
    def execute(self, action: str) -> str:
        self.log.append(action)
        if "click" in action:
            elem_id = action.split()[-1]
            elem = self.screen.find_element(elem_id.replace("btn_", ""))
            if elem:
                return f"Clicked '{elem.label}'"
            return f"Element '{elem_id}' not found"
        if "type" in action:
            parts = action.split()
            return f"Typed into '{parts[1]}'"
        return "Done"
    
    def run(self, goal: str, max_steps: int = 3):
        print(f"Goal: {goal}")
        print(f"Screen:\n{self.screen.render()}\n")
        for step in range(1, max_steps + 1):
            action = self.decide_action(goal)
            if action == "done": break
            result = self.execute(action)
            print(f"Step {step}: {action} → {result}")

agent = ToyGUIAgent(ToyScreen())
agent.run("create new file")

Goal: create new file
Screen:
[ File Manager ]
  [btn_new] New File @ (0,1)
  [btn_open] Open @ (0,12)
  [btn_delete] Delete @ (0,22)
  [input_search] Search... @ (2,1)
  [list_files] file1.txt
file2.pdf @ (4,1)

Step 1: click btn_new → Clicked 'New File'
Step 2: click btn_new → Clicked 'New File'
Step 3: click btn_new → Clicked 'New File'


## Computer Use 的安全挑战

传统 API Agent 的安全边界清晰（tool allowlist），但 Computer Use Agent 能看到一切：

- 屏幕上其他应用的弹出通知（含敏感信息）
- 密码管理器的自动填充提示
- 浏览器中已登录的 session
- 系统设置和权限对话框

> **检查点**：能列举 Computer Use Agent 特有的 3 个安全风险及其缓解方案。

## 真实 API 实验

文本版 Computer Use：把 ASCII 界面给真实 LLM，让它输出下一步操作坐标 — 与视觉版同构的决策循环。


In [3]:
# 文本 UI → LLM 决策 → 执行（Computer Use 的同构模拟）
screen = """
+----------------------------------+
|  MiniShop              [搜索 🔍] |
+----------------------------------+
|  (10,3) 📕 Python Cookbook  $39  |
|  (10,5) 📕 Deep Learning    $59  |
|  (10,7) 💻 DevBook Pro    $1299  |
+----------------------------------+
|  (2,9) [首页] (8,9) [购物车]     |
+----------------------------------+
"""

action = chat_json(client, f"""你在操作一个 GUI。当前屏幕（含元素坐标）：
{screen}
任务：把最便宜的 book 加入购物车。
返回下一步操作 JSON: {{"action": "click|type|scroll|done", "x": 数字, "y": 数字, "reason": "一句话"}}""")
print("LLM 决策:", action)


LLM 决策: {'action': 'click', 'x': 10, 'y': 3, 'reason': '点击最便宜的书籍Python Cookbook进入详情页以便加入购物车'}


## 练习

1. 给 ToyGUIAgent 增加视觉理解：解析更复杂的 UI 描述（含嵌套元素、动态菜单）。
2. 实现一个「回到主屏幕」的 fallback 机制：当 Agent 迷失时，执行标准恢复流程。
3. 设计 Computer Use Agent 的安全策略：哪些元素永远不可点击？哪些操作需要二次确认？